<a href="https://colab.research.google.com/github/avocado-planet/05-LangChain-Memory/blob/main/05_langchain_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain メモリコンポーネント

LangChain 1.0 のメモリ機構を **短期メモリ（Short-term）** と **長期メモリ（Long-term）** に分けて学びます。

## 目次

**Part 1: 短期メモリ**
1. セットアップ
2. 基本 — checkpointer で会話を記憶する
3. thread_id で会話を分離する
4. メッセージのトリミング（古いメッセージを切り捨て）
5. メッセージの削除
6. SummarizationMiddleware による自動要約

**Part 2: 長期メモリ**
7. Store の基本 — namespace と key
8. ツールから長期メモリを読み取る
9. ツールから長期メモリに書き込む
10. セッションをまたいだ記憶の確認

---
## 1. セットアップ

In [1]:
!pip install --pre -U langchain langgraph langchain-openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

MODEL_NAME = "openai:gpt-4.1-mini"
MODEL_NAME_CHEAP = "openai:gpt-4.1-nano"

print(f"モデル: {MODEL_NAME}")

モデル: openai:gpt-4.1-mini


---
# Part 1: 短期メモリ（Short-term Memory）

短期メモリ = **1つの会話（thread）内での記憶**。

仕組みは単純で、`checkpointer` がメッセージ履歴を保存し、
同じ `thread_id` で `invoke` するたびに前回の続きとして扱われます。

## 2. 基本 — checkpointer で会話を記憶する

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

# === checkpointer なしのエージェント ===
agent_no_memory = create_agent(
    model=MODEL_NAME,
    tools=[],
)

print("=== checkpointer なし ===")
r1 = agent_no_memory.invoke({"messages": [HumanMessage("私の名前はタロウです")]})
print(f"1回目: {r1['messages'][-1].content}")

r2 = agent_no_memory.invoke({"messages": [HumanMessage("私の名前を覚えていますか？")]})
print(f"2回目: {r2['messages'][-1].content}")
print("→ 覚えていない! 毎回リセットされる")

=== checkpointer なし ===
1回目: はじめまして、タロウさん！どうぞよろしくお願いします。今日はどんなことをお話ししましょうか？
2回目: 申し訳ありませんが、お名前を覚えていません。改めて教えていただけますか？
→ 覚えていない! 毎回リセットされる


In [4]:
# === checkpointer ありのエージェント ===
agent_with_memory = create_agent(
    model=MODEL_NAME,
    tools=[],
    checkpointer=InMemorySaver(),  # ★★★これを追加するだけ!
)

# thread_id で会話を識別する
config = {"configurable": {"thread_id": "memory-demo-1"}}

print("\n=== checkpointer あり ===")
r1 = agent_with_memory.invoke(
    {"messages": [HumanMessage("私の名前はタロウです")]},
    config=config,
)
print(f"1回目: {r1['messages'][-1].content}")

# 同じ thread_id で再度呼ぶ → 前回の会話を覚えている
r2 = agent_with_memory.invoke(
    {"messages": [HumanMessage("私の名前を覚えていますか？")]},
    config=config,  # 同じ thread_id!
)
print(f"2回目: {r2['messages'][-1].content}")
print("→ 覚えている! checkpointer がメッセージ履歴を保存しているため")


=== checkpointer あり ===
1回目: はじめまして、タロウさん！どうぞよろしくお願いします。何かお手伝いできることがあれば教えてくださいね。
2回目: はい、タロウさんのお名前を覚えていますよ！何かほかに知りたいことや話したいことがありますか？
→ 覚えている! checkpointer がメッセージ履歴を保存しているため


## 3. thread_id で会話を分離する

異なる `thread_id` は別の会話として扱われます。
ユーザーごと・セッションごとに分離できます。

In [5]:
# 会話A: タロウさん
config_a = {"configurable": {"thread_id": "user-taro"}}
agent_with_memory.invoke(
    {"messages": [HumanMessage("私はタロウです。猫が好きです。")]},
    config=config_a,
)

# 会話B: ハナコさん（別の thread_id）
config_b = {"configurable": {"thread_id": "user-hanako"}}
agent_with_memory.invoke(
    {"messages": [HumanMessage("私はハナコです。犬が好きです。")]},
    config=config_b,
)

# 会話Aに戻る → タロウさんの情報を覚えている
r = agent_with_memory.invoke(
    {"messages": [HumanMessage("私の好きな動物は？")]},
    config=config_a,
)
print(f"タロウの会話: {r['messages'][-1].content}")

# 会話Bに戻る → ハナコさんの情報を覚えている
r = agent_with_memory.invoke(
    {"messages": [HumanMessage("私の好きな動物は？")]},
    config=config_b,
)
print(f"ハナコの会話: {r['messages'][-1].content}")
print("\n→ thread_id が違うので、それぞれ独立した記憶を持っている")

タロウの会話: タロウさんの好きな動物は猫ですね。猫がお好きだとおっしゃっていました。何か猫についてもっとお話ししましょうか？
ハナコの会話: ハナコさんの好きな動物は犬ですね！犬についてもっと教えてもらえますか？

→ thread_id が違うので、それぞれ独立した記憶を持っている


## 4. メッセージのトリミング

会話が長くなるとコンテキストウィンドウを超えます。
`@before_model` ミドルウェアで古いメッセージを切り捨てます。

In [7]:
from langchain.agents.middleware import before_model, AgentState
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """最初のメッセージ + 最新3〜4件だけ残す。"""
    messages = state["messages"]
    if len(messages) <= 4:
        return None  # 短い会話はそのまま

    first_msg = messages[0]  # 最初のユーザーメッセージは残す
    # 最新の3〜4件を残す（結合後に user→assistant の交互順を守るため、偶数/奇数でペアを保つ）
    recent = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent

    print(f"  [trim] {len(messages)}件 → {len(new_messages)}件にトリミング")
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),  # 全削除
            *new_messages,  # 残すメッセージを再追加
        ]
    }


agent_trim = create_agent(
    model=MODEL_NAME,
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config_trim = {"configurable": {"thread_id": "trim-demo"}}

# 複数回の会話を重ねる
conversations = [
    "私の名前はタロウです",
    "好きな色は青です",
    "趣味はプログラミングです",
    "今日は天気がいいですね",
    "最近Pythonを勉強しています",
    "私の名前を覚えていますか？",  # トリミングされても覚えているか?
    "私の好きな色を覚えていますか？" # 残念ですが、会話を切り捨てたため、覚えていない。
]

for msg in conversations:
    print(f"\nユーザー: {msg}")
    r = agent_trim.invoke(
        {"messages": [HumanMessage(msg)]},
        config=config_trim,
    )
    print(f"AI: {r['messages'][-1].content[:100]}")


ユーザー: 私の名前はタロウです
AI: こんにちは、タロウさん！はじめまして。今日はどういったご用件でしょうか？何かお手伝いできることがあれば教えてくださいね。

ユーザー: 好きな色は青です
AI: 青が好きなんですね！青は落ち着いた感じがあって素敵な色ですよね。ほかにも好きなものや趣味はありますか？

ユーザー: 趣味はプログラミングです
  [trim] 5件 → 5件にトリミング
AI: プログラミングが趣味なんですね！素晴らしいです。どんな言語をよく使いますか？また、どんなプロジェクトや作りたいものがありますか？

ユーザー: 今日は天気がいいですね
  [trim] 7件 → 5件にトリミング
AI: はい、今日は本当に天気が良くて気持ちいいですね！こんな日は外でリフレッシュするのも良さそうです。タロウさんは天気のいい日は何をするのが好きですか？

ユーザー: 最近Pythonを勉強しています
  [trim] 7件 → 5件にトリミング
AI: いいですね、Pythonはとても人気があって使いやすい言語です！どんな分野に興味がありますか？例えば、データ分析、ウェブ開発、機械学習など。また、何か作ってみたいプロジェクトがあれば教えてください！

ユーザー: 私の名前を覚えていますか？
  [trim] 7件 → 5件にトリミング
AI: はい、タロウさんですね！覚えていますよ。何か他にお手伝いできることがあれば教えてください。

ユーザー: 私の好きな色を覚えていますか？
  [trim] 7件 → 5件にトリミング
AI: 申し訳ありませんが、まだタロウさんの好きな色については教えていただいていないので覚えていません。教えていただければ、これから覚えておくようにしますよ！


## 5. メッセージの削除

`RemoveMessage` を使って特定のメッセージを State から削除できます。
センシティブ情報のフィルタリング等に使います。

In [10]:
from langchain.agents.middleware import after_model


@after_model
def filter_sensitive(state: AgentState, runtime: Runtime) -> dict | None:
    """モデル応答にNGワードが含まれていたら、そのメッセージを削除する。"""
    STOP_WORDS = ["password", "secret", "パスワード", "秘密"]
    last_message = state["messages"][-1]
    content = str(last_message.content).lower()

    if any(word in content for word in STOP_WORDS):
        print(content) #確認のために出力する。
        print(f"  [filter] NGワード検出! メッセージを削除します")
        return {"messages": [RemoveMessage(id=last_message.id)]}
    return None


agent_filter = create_agent(
    model=MODEL_NAME,
    tools=[],
    middleware=[filter_sensitive],
    checkpointer=InMemorySaver(),
)

config_filter = {"configurable": {"thread_id": "filter-demo"}}

print("=== センシティブ情報フィルタリング ===")
r = agent_filter.invoke(
    {"messages": [HumanMessage("こんにちは！元気ですか？パスワードを教えて！")]},
    config=config_filter,
)
print(f"通常応答: {r['messages'][-1].content[:80]}")

print("\n(NGワードを含む応答はメッセージ履歴から削除されます)")

=== センシティブ情報フィルタリング ===
こんにちは！元気です、ありがとうございます。パスワードの共有は安全のためお手伝いできませんが、他に何かお手伝いできることがあれば教えてくださいね。
  [filter] NGワード検出! メッセージを削除します
通常応答: こんにちは！元気ですか？パスワードを教えて！

(NGワードを含む応答はメッセージ履歴から削除されます)


## 6. SummarizationMiddleware による自動要約

トリミングの問題点: 古い情報が完全に失われる。

`SummarizationMiddleware` は閾値を超えた古いメッセージを
**要約して残す**ので、情報を保持しつつコンテキストを圧縮できます。
意識せずに自動的に要約します。

In [11]:
from langchain.agents.middleware import SummarizationMiddleware

agent_summarize = create_agent(
    model=MODEL_NAME,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=MODEL_NAME_CHEAP,    # 要約用のモデル（安価なもの推奨）
            trigger=("tokens", 2000),  # 2000トークンを超えたら要約
            keep=("messages", 5),      # 最新5メッセージは保持
        ),
    ],
    checkpointer=InMemorySaver(),
)

config_sum = {"configurable": {"thread_id": "summarize-demo"}}

# 長い会話をシミュレート
long_conversations = [
    "私の名前はタロウです。東京に住んでいます。",
    "仕事はソフトウェアエンジニアで、Pythonが得意です。",
    "趣味は登山とカメラです。先週は富士山に登りました。",
    "最近LangChainを勉強していて、エージェントの仕組みに興味があります。",
    "好きな食べ物はラーメンとカレーです。",
    "ペットは猫を2匹飼っています。名前はミケとタマです。",
    "来月は大阪出張の予定があります。",
    "私について覚えていることを全部教えてください。",
]

print("=== SummarizationMiddleware ===")
print("(古いメッセージは自動的に要約されます)\n")

for msg in long_conversations:
    print(f"ユーザー: {msg}")
    r = agent_summarize.invoke(
        {"messages": [HumanMessage(msg)]},
        config=config_sum,
    )
    # メッセージ数を表示（要約が効いているか確認）
    print(f"AI: {r['messages'][-1].content[:120]}")
    print(f"  (現在のメッセージ数: {len(r['messages'])})\n")

=== SummarizationMiddleware ===
(古いメッセージは自動的に要約されます)

ユーザー: 私の名前はタロウです。東京に住んでいます。
AI: はじめまして、タロウさん。東京にお住まいなんですね。何かお手伝いできることがあれば教えてください。
  (現在のメッセージ数: 2)

ユーザー: 仕事はソフトウェアエンジニアで、Pythonが得意です。
AI: タロウさん、ソフトウェアエンジニアでPythonが得意なんですね！素晴らしいです。Pythonに関して何か質問や相談があれば、ぜひ教えてください。コードのアドバイスやプロジェクトのアイデアなど、何でもお手伝いしますよ。
  (現在のメッセージ数: 4)

ユーザー: 趣味は登山とカメラです。先週は富士山に登りました。
AI: 富士山に登られたんですね！素晴らしい経験ですね。登山とカメラが趣味だと、自然の美しい景色をたくさん写真に収められて楽しそうです。富士山で撮った写真のお気に入りはありますか？もしよければ、登山のエピソードや写真の話も聞かせてください。
  (現在のメッセージ数: 6)

ユーザー: 最近LangChainを勉強していて、エージェントの仕組みに興味があります。
AI: LangChainに興味を持って勉強されているんですね！エージェントの仕組みはとても面白い分野です。

LangChainのエージェントは、ユーザーの指示や質問に応じて、自動的に適切なツールやAPIを選択し、タスクを実行する仕組みです。例え
  (現在のメッセージ数: 8)

ユーザー: 好きな食べ物はラーメンとカレーです。
AI: ラーメンとカレーがお好きなんですね！どちらも日本でも人気のある美味しい料理ですよね。特に東京にはたくさんの美味しいラーメン屋さんやカレー屋さんがありますが、お気に入りのお店やおすすめの味などはありますか？
  (現在のメッセージ数: 10)

ユーザー: ペットは猫を2匹飼っています。名前はミケとタマです。
AI: ミケちゃんとタマちゃん、かわいい名前ですね！猫を2匹飼っているなんて、きっと賑やかで楽しい毎日でしょうね。ミケちゃんとタマちゃんの性格や好きな遊びなど、もしよければ教えてください。猫との生活のお話、ぜひ聞かせてくださいね。
  (現在のメッセージ数: 12)

ユ

---
# Part 2: 長期メモリ（Long-term Memory）

長期メモリ = **会話（thread）をまたいで持続する記憶**。

短期メモリ（checkpointer）は1つの thread 内でしか有効ではありません。
別の thread_id で新しい会話を始めると、以前の情報は見えません。

長期メモリは **Store** という仕組みで、
namespace（フォルダ）と key（ファイル名）でデータを整理して保存します。

```
短期メモリ（checkpointer）     長期メモリ（Store）
──────────────────────     ────────────────────
thread 単位                 thread をまたぐ
メッセージ履歴              任意のJSONデータ
自動的に保存               ツールで明示的に読み書き
InMemorySaver / DB         InMemoryStore / DB
```

## 7. Store の基本 — namespace と key

In [12]:
from langgraph.store.memory import InMemoryStore

# Store を作成
store = InMemoryStore()

# === データの保存 ===
# namespace: タプルで階層構造を表現（フォルダのようなもの）
# key: そのnamespace内での識別子（ファイル名のようなもの）
store.put(
    ("users",),        # namespace
    "user_001",        # key
    {                  # value（任意のJSON）
        "name": "タロウ",
        "language": "Japanese",
        "interests": ["Python", "LangChain", "登山"],
    },
)

store.put(
    ("users",),
    "user_002",
    {
        "name": "ハナコ",
        "language": "Japanese",
        "interests": ["React", "TypeScript", "料理"],
    },
)

# namespace を階層化することも可能
store.put(
    ("users", "user_001", "preferences"),  # 深い階層
    "display",
    {"theme": "dark", "language": "ja"},
)

# === データの読み取り ===
item = store.get(("users",), "user_001")
print(f"取得: {item.value}")

# === データの検索 ===
items = store.search(("users",))  # namespace 内の全データ
print(f"\nusers namespace 内のデータ:")
for item in items:
    print(f"  key={item.key}, value={item.value}")

取得: {'name': 'タロウ', 'language': 'Japanese', 'interests': ['Python', 'LangChain', '登山']}

users namespace 内のデータ:
  key=user_001, value={'name': 'タロウ', 'language': 'Japanese', 'interests': ['Python', 'LangChain', '登山']}
  key=user_002, value={'name': 'ハナコ', 'language': 'Japanese', 'interests': ['React', 'TypeScript', '料理']}
  key=display, value={'theme': 'dark', 'language': 'ja'}


## 8. ツールから長期メモリを読み取る

エージェントのツールから `runtime.store` 経由で Store にアクセスします。
`runtime` パラメータはモデルからは見えない（hidden）ので、
ツールのシグネチャに含めても LLM が混乱しません。

In [16]:
from langchain_core.tools import tool
from langchain.tools import ToolRuntime
from dataclasses import dataclass


# Context: invoke 時に渡す追加情報（user_id など）
@dataclass
class UserContext:
    user_id: str


# === 長期メモリから読み取るツール ===
@tool
def get_user_profile(runtime: ToolRuntime[UserContext]) -> str:
    """ユーザーのプロフィール情報を取得します。"""
    assert runtime.store is not None
    user_id = runtime.context.user_id
    item = runtime.store.get(("users",), user_id)
    if item:
        return f"ユーザー情報: {item.value}"
    return "ユーザー情報が見つかりません"


# Store にデータを事前投入
store_read = InMemoryStore()
store_read.put(("users",), "user_001", {
    "name": "タロウ",
    "interests": ["Python", "登山"],
    "location": "東京",
})

# エージェント作成（store を渡す）
agent_read = create_agent(
    model=MODEL_NAME,
    tools=[get_user_profile],
    store=store_read,           # 長期メモリ
    context_schema=UserContext,  # context の型
)

print("=== 長期メモリ読み取り ===")
result = agent_read.invoke(
    {"messages": [HumanMessage("私のプロフィールを教えて")]},
    context=UserContext(user_id="user_001"),
)
print(f"応答: {result['messages'][-1].content}")

=== 長期メモリ読み取り ===
応答: あなたのプロフィールは以下の通りです。
名前: タロウ
興味: Python, 登山
居住地: 東京
他に知りたいことがあれば教えてください。


## 9. ツールから長期メモリに書き込む

会話中にユーザーが話した情報を長期メモリに保存します。
次回の会話（別の thread）でもその情報を使えるようになります。

In [17]:
from typing_extensions import TypedDict


class UserInfo(TypedDict):
    name: str
    interests: list[str]


# === 長期メモリに書き込むツール ===
@tool
def save_user_info(user_info: UserInfo, runtime: ToolRuntime[UserContext]) -> str:
    """ユーザー情報を保存します。名前や趣味を記録してください。"""
    assert runtime.store is not None
    user_id = runtime.context.user_id
    runtime.store.put(("users",), user_id, dict(user_info))
    return f"ユーザー情報を保存しました: {user_info}"


# === 長期メモリから読み取るツール（再掲） ===
@tool
def lookup_user(runtime: ToolRuntime[UserContext]) -> str:
    """保存済みのユーザー情報を取得します。"""
    assert runtime.store is not None
    user_id = runtime.context.user_id
    item = runtime.store.get(("users",), user_id)
    if item:
        return f"保存済み情報: {item.value}"
    return "まだユーザー情報が保存されていません"


# Store（空の状態から開始）
store_rw = InMemoryStore()

agent_rw = create_agent(
    model=MODEL_NAME,
    tools=[save_user_info, lookup_user],
    store=store_rw,
    context_schema=UserContext,
    checkpointer=InMemorySaver(),  # 短期メモリも併用
)

user_ctx = UserContext(user_id="user_taro")

print("=== 会話1: 情報を保存 ===")
r = agent_rw.invoke(
    {"messages": [HumanMessage(
        "私の名前はタロウです。趣味はPythonと登山です。覚えておいてください。"
    )]},
    config={"configurable": {"thread_id": "session-1"}},
    context=user_ctx,
)
print(f"応答: {r['messages'][-1].content}")

# Store の中身を直接確認
stored = store_rw.get(("users",), "user_taro")
print(f"\nStore に保存された値: {stored.value if stored else 'なし'}")

=== 会話1: 情報を保存 ===
応答: タロウさん、趣味のPythonと登山について覚えました。何かそれに関連してお手伝いできることがあれば教えてくださいね。

Store に保存された値: {'name': 'タロウ', 'interests': ['Python', '登山']}


## 10. セッションをまたいだ記憶の確認

**別の thread_id**（= 新しい会話）でも、
長期メモリ（Store）に保存した情報は取得できます。

In [18]:
print("=== 会話2: 新しいセッションで情報を取得 ===")
print("(thread_id が違う = 短期メモリはリセット)")
print("(同じ user_id  = 長期メモリは維持)\n")

r = agent_rw.invoke(
    {"messages": [HumanMessage(
        "私の保存されている情報を教えてください"
    )]},
    config={"configurable": {"thread_id": "session-2"}},  # 別の thread!
    context=user_ctx,  # 同じ user_id
)
print(f"応答: {r['messages'][-1].content}")
print("\n→ 別のセッションでも長期メモリの情報を取得できた!")

=== 会話2: 新しいセッションで情報を取得 ===
(thread_id が違う = 短期メモリはリセット)
(同じ user_id  = 長期メモリは維持)

応答: あなたの保存されている情報は以下の通りです。
名前: タロウ
趣味・興味: Python、登山

他に知りたいことがあれば教えてください。

→ 別のセッションでも長期メモリの情報を取得できた!


---
## まとめ

| 項目 | 短期メモリ | 長期メモリ |
|---|---|---|
| スコープ | 1つの thread（会話）内 | thread をまたぐ |
| 保存対象 | メッセージ履歴 | 任意の JSON データ |
| 仕組み | checkpointer + thread_id | Store (namespace + key) |
| 保存タイミング | 自動（invoke のたびに） | 明示的（ツールから put） |
| 開発用 | `InMemorySaver()` | `InMemoryStore()` |
| 本番用 | PostgresSaver 等 | PostgresStore 等 |
| コンテキスト管理 | trim / delete / summarize | namespace で整理 |